In [3]:
from __future__ import annotations
import os
import re
import io
import json
import csv
import itertools
import mimetypes
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Iterator, Union
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
import argparse
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from collections import Counter
import time
import sys
import warnings
import logging
import urllib.request
import hashlib
import numpy as np

# Настройка кодировки для Windows
if sys.platform == 'win32':
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
        sys.stderr.reconfigure(encoding='utf-8', errors='replace')
    except:
        pass

# Подавляем все предупреждения
warnings.filterwarnings('ignore')
logging.getLogger('PyPDF2').setLevel(logging.ERROR)
logging.getLogger('pdfminer').setLevel(logging.ERROR)
logging.getLogger('PIL').setLevel(logging.ERROR)
logging.getLogger('ultralytics').setLevel(logging.ERROR)

# Подавляем вывод Ultralytics
os.environ['ULTRALYTICS_VERBOSE'] = 'False'
os.environ['YOLO_VERBOSE'] = 'False'

# ============================================================================
# Определение GPU доступности
# ============================================================================

def is_gpu_available():
    """Проверка доступности GPU"""
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"CUDA ({torch.cuda.get_device_name(0)})"
        elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            return True, "MPS (Apple Silicon)"
    except:
        pass
    
    try:
        import tensorflow as tf
        if tf.config.list_physical_devices('GPU'):
            return True, "TensorFlow GPU"
    except:
        pass
    
    return False, "CPU"

GPU_AVAILABLE, GPU_NAME = is_gpu_available()

# ============================================================================
# Конфигурации и константы
# ============================================================================

# Включаем все форматы, включая изображения
INCLUDE_EXTS = {'doc', 'docx', 'gif', 'html', 'ipynb', 'jpeg', 'jpg', 'pdf', 
                'php', 'png', 'rtf', 'xls', 'xlsx', 'txt', 'csv', 'json', 'tif', 'tiff', 'bmp'}

PDN_CATEGORIES = ['обычные', 'государственные', 'платёжные', 'биометрические', 'специальные']

# ============================================================================
# Функция получения даты создания файла с корректировкой на -2 часа
# ============================================================================

def get_file_creation_time_minus_2h(file_path: str) -> str:
    """
    Получение времени изменения файла (Изменен / st_mtime) минус 2 часа
    в формате 'sep 26 18:31'. Работает на Windows, macOS и Linux.
    """
    try:
        stat = os.stat(file_path)
        
        # Время последнего изменения файла (поле «Изменен»)
        mtime = stat.st_mtime
        
        # Вычитаем 2 часа (7200 секунд)
        mtime = mtime - 7200
        
        # Форматируем дату: месяц (сокращенно), день, часы:минуты
        dt = datetime.fromtimestamp(mtime)
        
        # Месяцы на английском (сокращенно)
        months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 
                  'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
        month_abbr = months[dt.month - 1]
        
        return f"{month_abbr} {dt.day:02d} {dt.hour:02d}:{dt.minute:02d}"
    except:
        return 'unknown'

# ============================================================================
# YOLO инициализация с автоматической загрузкой моделей
# ============================================================================

YOLO_AVAILABLE = False
_yolo_model = None
_yolo_lock = threading.Lock()
_yolo_initialized = False

# URL для скачивания моделей
MODEL_URLS = {
    'yolov8n.pt': 'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8n.pt',
}

def download_model(model_name: str, model_path: str) -> bool:
    """Скачивание модели с GitHub"""
    if model_name not in MODEL_URLS:
        return False
    
    url = MODEL_URLS[model_name]
    print(f"Скачивание модели {model_name}...")
    
    try:
        urllib.request.urlretrieve(url, model_path)
        print(f"✅ Модель {model_name} успешно скачана")
        return True
    except Exception as e:
        print(f"⚠️ Не удалось скачать {model_name}: {e}")
        return False

try:
    import contextlib
    with contextlib.redirect_stdout(open(os.devnull, 'w')):
        from ultralytics import YOLO
        import cv2
        import numpy as np
        from PIL import Image
    YOLO_AVAILABLE = True
    
    def get_yolo_model():
        """Потокобезопасная загрузка YOLO модели (один раз)"""
        global _yolo_model, _yolo_initialized
        
        if _yolo_initialized:
            return _yolo_model
        
        with _yolo_lock:
            if _yolo_initialized:
                return _yolo_model
            
            try:
                device = 'cuda' if GPU_AVAILABLE and 'CUDA' in GPU_NAME else ('mps' if GPU_AVAILABLE and 'MPS' in GPU_NAME else 'cpu')
                print(f"Загрузка YOLO модели на устройство: {device}")
                
                model_name = 'yolov8n.pt'
                model_path = model_name
                
                if not os.path.exists(model_path):
                    if download_model(model_name, model_path):
                        print(f"✅ Модель {model_name} загружена")
                    else:
                        print(f"⚠️ Не удалось загрузить модель")
                        return None
                
                with contextlib.redirect_stdout(open(os.devnull, 'w')):
                    model = YOLO(model_path)
                
                if model:
                    print(f"✅ Загружена модель: {model_name}")
                    
                    if device != 'cpu':
                        try:
                            with contextlib.redirect_stdout(open(os.devnull, 'w')):
                                model.to(device)
                            print(f"✅ Модель перемещена на {device}")
                        except Exception as e:
                            print(f"⚠️ Не удалось переместить на GPU: {e}")
                else:
                    print("⚠️ Не удалось загрузить модель YOLO")
                    return None
                
                _yolo_model = model
                _yolo_initialized = True
                return _yolo_model
                
            except Exception as e:
                print(f"⚠️ Критическая ошибка загрузки YOLO: {e}")
                return None
    
except ImportError as e:
    print(f"Ultralytics YOLO не установлен. Установите: pip install ultralytics")
    def get_yolo_model():
        return None

# ============================================================================
# Вспомогательные функции
# ============================================================================

def luhn_check(number: str) -> bool:
    digits = [int(d) for d in re.sub(r'\D', '', number)]
    if not (13 <= len(digits) <= 19):
        return False
    s = 0
    parity = len(digits) % 2
    for i, d in enumerate(digits):
        if i % 2 == parity:
            d *= 2
            if d > 9:
                d -= 9
        s += d
    return s % 10 == 0

def snils_valid(snils: str) -> bool:
    nums = re.sub(r'\D', '', snils)
    if len(nums) != 11:
        return False
    base = [int(x) for x in nums[:9]]
    check = int(nums[9:])
    s = sum((9 - i) * d for i, d in enumerate(base))
    if s < 100:
        c = s
    elif s in (100, 101):
        c = 0
    else:
        c = s % 101
        if c == 100:
            c = 0
    return c == check

def inn_valid(inn: str) -> bool:
    nums = re.sub(r'\D', '', inn)
    if len(nums) == 10:
        w = [2, 4, 10, 3, 5, 9, 4, 6, 8]
        c = sum(int(nums[i]) * w[i] for i in range(9)) % 11 % 10
        return c == int(nums[9])
    elif len(nums) == 12:
        w1 = [7, 2, 4, 10, 3, 5, 9, 4, 6, 8, 0]
        w2 = [3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8, 0]
        c1 = sum(int(nums[i]) * w1[i] for i in range(11)) % 11 % 10
        c2 = sum(int(nums[i]) * w2[i] for i in range(11)) % 11 % 10
        return c1 == int(nums[10]) and c2 == int(nums[11])
    return False

def has_context(text: str, idx: int, window: int, *keywords: str) -> bool:
    start = max(0, idx - window)
    end = min(len(text), idx + window)
    chunk = text[start:end].lower()
    return any(k.lower() in chunk for k in keywords)

# ============================================================================
# Детекция лиц с помощью YOLO на GPU (потокобезопасная)
# ============================================================================

_yolo_inference_lock = threading.Lock()

def detect_faces_in_image_with_model(image_bytes: bytes, model) -> int:
    """Детекция лиц - возвращает только количество лиц (потокобезопасная)"""
    if model is None:
        return 0
    
    try:
        nparr = np.frombuffer(image_bytes, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        
        if img is None:
            return 0
        
        height, width = img.shape[:2]
        if width > 640 or height > 640:
            scale = min(640/width, 640/height)
            new_width = int(width * scale)
            new_height = int(height * scale)
            img = cv2.resize(img, (new_width, new_height))
        
        with _yolo_inference_lock:
            with contextlib.redirect_stdout(open(os.devnull, 'w')):
                results = model(img, conf=0.25, verbose=False)
            
            face_count = 0
            for result in results:
                boxes = result.boxes
                if boxes is not None:
                    face_count = len(boxes)
            
            return face_count
        
    except Exception:
        return 0

# ============================================================================
# Извлечение текста из файлов
# ============================================================================

def extract_text_from_bytes(content: bytes, ext: str, detect_faces: bool = True, yolo_model=None) -> Tuple[str, int]:
    face_count = 0
    
    try:
        if ext in {'txt', 'csv', 'json', 'log', 'md', 'py', 'js', 'html', 'xml', 'php', 'rtf'}:
            for enc in ['utf-8', 'cp1251', 'latin-1']:
                try:
                    return content.decode(enc, errors='ignore'), 0
                except:
                    continue
            return content.decode('utf-8', errors='ignore'), 0
        
        elif ext == 'pdf':
            text = ''
            with contextlib.redirect_stderr(open(os.devnull, 'w')):
                try:
                    import PyPDF2
                    reader = PyPDF2.PdfReader(io.BytesIO(content))
                    for page in reader.pages[:10]:
                        try:
                            page_text = page.extract_text()
                            if page_text:
                                text += page_text + '\n'
                        except:
                            pass
                    if text.strip():
                        return text, 0
                except:
                    pass
                
                try:
                    from pdfminer.high_level import extract_text as pdfminer_extract
                    text = pdfminer_extract(io.BytesIO(content)) or ''
                    if text.strip():
                        return text, 0
                except:
                    pass
            
            return '', 0
        
        elif ext in {'xls', 'xlsx', 'xlsm'}:
            try:
                df = pd.read_excel(io.BytesIO(content), header=None, dtype=str, nrows=1000)
                return ' '.join(df.stack().astype(str).tolist()), 0
            except:
                return '', 0
        
        elif ext == 'docx':
            try:
                import docx
                doc = docx.Document(io.BytesIO(content))
                parts = []
                for p in doc.paragraphs:
                    if p.text:
                        parts.append(p.text)
                for tbl in doc.tables:
                    for row in tbl.rows:
                        row_text = ' \t '.join(cell.text for cell in row.cells if cell.text)
                        if row_text:
                            parts.append(row_text)
                return '\n'.join(parts), 0
            except:
                return '', 0
        
        elif ext == 'doc':
            try:
                text = content.decode('latin-1', errors='ignore')
                text = re.sub(r'[^\x20-\x7E\u0400-\u04FF]+', ' ', text)
                return re.sub(r'\s+', ' ', text), 0
            except:
                return '', 0
        
        elif ext == 'html':
            try:
                txt = content.decode('utf-8', errors='ignore')
                from bs4 import BeautifulSoup
                soup = BeautifulSoup(txt, 'html.parser')
                return soup.get_text(' '), 0
            except:
                return content.decode('utf-8', errors='ignore'), 0
        
        elif ext == 'ipynb':
            try:
                data = json.loads(content.decode('utf-8', errors='ignore'))
                parts = []
                for cell in data.get('cells', []):
                    src = cell.get('source', [])
                    if isinstance(src, list):
                        parts.append(''.join(src))
                    elif isinstance(src, str):
                        parts.append(src)
                return '\n'.join(parts), 0
            except:
                return '', 0
        
        elif ext in {'jpg', 'jpeg', 'png', 'gif', 'tif', 'tiff', 'bmp'}:
            text = ''
            try:
                import pytesseract
                from PIL import Image, ImageEnhance, ImageFilter
                img = Image.open(io.BytesIO(content))
                if img.mode not in ('RGB', 'L'):
                    img = img.convert('RGB')
                if img.width * img.height > 4000000:
                    img.thumbnail((2000, 2000))
                
                img = img.convert('L')
                img = img.filter(ImageFilter.SHARPEN)
                enhancer = ImageEnhance.Contrast(img)
                img = enhancer.enhance(1.5)
                
                text = pytesseract.image_to_string(img, lang='rus+eng')
            except:
                pass
            
            if detect_faces and yolo_model:
                face_count = detect_faces_in_image_with_model(content, yolo_model)
            
            # Запасной метод детекции лиц через OpenCV Haar Cascade (без YOLO)
            if face_count == 0:
                try:
                    import cv2
                    # cv2.imdecode не поддерживает TIFF — конвертируем через PIL
                    if ext in {"tif", "tiff"}:
                        pil_img = Image.open(io.BytesIO(content)).convert("L")
                        cv_img = np.array(pil_img)
                    else:
                        nparr = np.frombuffer(content, np.uint8)
                        cv_img = cv2.imdecode(nparr, cv2.IMREAD_GRAYSCALE)
                    if cv_img is not None:
                        cascade = cv2.CascadeClassifier(
                            cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
                        faces = cascade.detectMultiScale(
                            cv_img, scaleFactor=1.1, minNeighbors=4, minSize=(30, 30))
                        face_count = len(faces) if len(faces) > 0 else 0
                except:
                    pass
            
            return text, face_count
        
        else:
            return content.decode('utf-8', errors='ignore'), 0
            
    except Exception:
        return '', 0

# ============================================================================
# Детекторы ПДн
# ============================================================================

EMAIL_RE = re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"(?:(?:\+7|8)\s*\(?\d{3}\)?[\s-]?\d{3}[\s-]?\d{2}[\s-]?\d{2})")
FIO_RE = re.compile(r"\b[А-ЯЁ][а-яё]+\s+[А-ЯЁ][а-яё]+(?:\s+[А-ЯЁ][а-яё]+)?\b")
# Латинские имена (иностранные документы, ID-карты)
FIO_LATIN_RE = re.compile(r"\b[A-Z][a-z]{2,}\s+[A-Z][a-z]{2,}(?:\s+[A-Z][a-z]{2,})?\b")
# Дата: точка, слеш или дефис (01-01-1980 / 01.01.1980 / 01/01/1980)
DOB_RE = re.compile(r"\b(\d{2}[.\-/]\d{2}[.\-/]\d{4})\b")
# Номер иностранного ID/паспорта: буква + цифры (I05101999Q, AB1234567)
ID_CARD_RE = re.compile(r"\b[A-Z]\d{7,10}[A-Z0-9]?\b")
INDEX_RE = re.compile(r"\b\d{6}\b")
SNILS_RE = re.compile(r"\b\d{3}-\d{3}-\d{3}\s?\d{2}\b|\b\d{11}\b")
INN10_RE = re.compile(r"(?<!\d)\d{10}(?!\d)")
INN12_RE = re.compile(r"(?<!\d)\d{12}(?!\d)")
PASSPORT_RE = re.compile(r"(?:(?<!\d)\d{2}\s?\d{2}\s?\d{6}(?!\d))")
CARD_RE = re.compile(r"(?:(?:\d[ -]*?){13,19})")
RS_RE = re.compile(r"(?i)(?:р/с|расч[её]тн(?:ый)?\s+сч[её]т)[^\d]*(\d{20})")
BIK_RE = re.compile(r"(?i)бик[^\d]*(\d{9})")
CVV_RE = re.compile(r"\b(CVV|CVC|CVV2)\b", re.IGNORECASE)

BIOMETRIC_KEYS = [
    'биометр', 'отпечат', 'радуж', 'ирис', 'лицев', 'селфи', 'faceid', 
    'fingerprint', 'iris', 'voiceprint', 'голосов', 'геометрия лица'
]

SPECIAL_KEYS = [
    'диагноз', 'анамнез', 'инвалид', 'здоровь', 'медицин', 'психиатр', 
    'вич', 'религ', 'вероисповед', 'политическ', 'партия', 'интим', 'сексуаль'
]

def count_occurrences(pattern: re.Pattern, text: str) -> int:
    return len(list(pattern.finditer(text)))

def detect_categories(text: str, face_count: int = 0) -> Dict[str, int]:
    t = text if isinstance(text, str) else ''
    low = t.lower()
    cats = {
        'обычные': 0,
        'государственные': 0,
        'платёжные': 0,
        'биометрические': 0,
        'специальные': 0
    }
    
    if face_count > 0:
        cats['биометрические'] += face_count
    
    if not t:
        return cats
    
    cats['обычные'] += count_occurrences(EMAIL_RE, t)
    cats['обычные'] += count_occurrences(PHONE_RE, t)
    cats['обычные'] += min(5, count_occurrences(FIO_RE, t))
    # Латинские имена (иностранные документы)
    cats['обычные'] += min(5, count_occurrences(FIO_LATIN_RE, t))
    
    for m in DOB_RE.finditer(t):
        if has_context(low, m.start(), 60, 'дата рождения', 'родил', 'д.р.', 'birth',
                       'date of birth', 'dob', 'born', 'lindur', 'datëlindja'):
            cats['обычные'] += 1
    
    for m in INDEX_RE.finditer(t):
        if has_context(low, m.start(), 40, 'ул', 'улица', 'просп', 'пер', 'дом', 'квартира', 'город', 'г.', 'адрес'):
            cats['обычные'] += 1
    
    for m in SNILS_RE.finditer(t):
        if snils_valid(m.group(0)):
            cats['государственные'] += 1
    
    for m in INN10_RE.finditer(t):
        if inn_valid(m.group(0)):
            cats['государственные'] += 1
    
    for m in INN12_RE.finditer(t):
        if inn_valid(m.group(0)):
            cats['государственные'] += 1
    
    for m in PASSPORT_RE.finditer(t):
        if has_context(low, m.start(), 50, 'паспорт', 'серия', 'номер', 'код подразделения',
                       'passport', 'document', 'id card', 'identity', 'nr.'):
            cats['государственные'] += 1
    
    # Иностранные ID/паспорта (напр. I05101999Q на Albanian ID)
    for m in ID_CARD_RE.finditer(t):
        if has_context(low, m.start(), 80, 'passport', 'id', 'identity', 'personal no',
                       'nr.', 'document', 'republic', 'паспорт', 'удостовер'):
            cats['государственные'] += 1
    
    for m in CARD_RE.finditer(t):
        raw = m.group(0)
        digits = re.sub(r'\D', '', raw)
        if 13 <= len(digits) <= 19 and luhn_check(raw):
            if has_context(low, m.start(), 40, 'visa', 'mastercard', 'карта', 'cvv', 'cvc', 'номер карты'):
                cats['платёжные'] += 1
    
    for m in RS_RE.finditer(t):
        cats['платёжные'] += 1
    
    for m in BIK_RE.finditer(t):
        cats['платёжные'] += 1
    
    if CVV_RE.search(t):
        cats['платёжные'] += 1
    
    if any(k in low for k in BIOMETRIC_KEYS):
        cats['биометрические'] += 1
    
    if any(k in low for k in SPECIAL_KEYS):
        cats['специальные'] += 1
    
    return cats

def estimate_uz(cats: Dict[str, int]) -> str:
    total = sum(cats.values())
    distinct = sum(1 for v in cats.values() if v > 0)
    has_special = cats['специальные'] > 0
    has_bio = cats['биометрические'] > 0
    has_pay = cats['платёжные'] > 0
    has_gov = cats['государственные'] > 0
    has_common = cats['обычные'] > 0
    
    if has_special or has_bio:
        return 'УЗ-1' if (total >= 5 or distinct >= 2) else 'УЗ-2'
    if has_pay or has_gov:
        return 'УЗ-2' if (total >= 5 or distinct >= 2) else 'УЗ-3'
    if has_common:
        return 'УЗ-3' if (total >= 5 or distinct >= 2) else 'УЗ-4'
    return 'нет ПДн'

# ============================================================================
# Анализ файла
# ============================================================================

def analyze_file_content(file_path: str, content: bytes, detect_faces: bool = True, yolo_model=None) -> Dict[str, Any]:
    path = Path(file_path)
    
    # Получаем размер файла и дату создания (минус 2 часа)
    try:
        file_size = os.path.getsize(file_path)
        file_ctime = get_file_creation_time_minus_2h(file_path)
    except:
        file_size = 0
        file_ctime = 'unknown'
    
    result = {
        'путь': str(file_path),
        'категории_ПДн': 'нет',
        'количество_находок': 0,
        'УЗ': 'нет ПДн',
        'формат_файла': '',
        'success': False,
        'error': '',
        'size': int(file_size) if file_size is not None else 0,
        'time': str(file_ctime) if file_ctime not in (None, '', 'nan', 'None') else 'unknown',
        'name': str(path.name) if path.name else 'unknown'
    }
    
    if content is None:
        result['error'] = 'Пустое содержимое'
        return result
    
    ext = path.suffix.lower().lstrip('.')
    result['формат_файла'] = ext
    
    if ext not in INCLUDE_EXTS:
        result['error'] = f'Неподдерживаемый формат: {ext}'
        return result
    
    try:
        text, face_count = extract_text_from_bytes(content, ext, detect_faces, yolo_model)
        
        cats = detect_categories(text, face_count)
        
        categories_list = [f"{k}:{v}" for k, v in cats.items() if v > 0]
        categories_str = ', '.join(categories_list) if categories_list else 'нет'
        
        result['категории_ПДн'] = categories_str
        result['количество_находок'] = sum(cats.values())
        result['УЗ'] = estimate_uz(cats)
        result['success'] = True
        
    except Exception as e:
        result['error'] = str(e)[:200]
        result['категории_ПДн'] = 'ошибка'
        result['количество_находок'] = 0
        result['УЗ'] = 'нет ПДн'
    
    # Гарантируем отсутствие NaN
    for key in result:
        if result[key] is None or (isinstance(result[key], float) and pd.isna(result[key])):
            if key in ['количество_находок', 'size']:
                result[key] = 0
            else:
                result[key] = ''
    
    return result

# ============================================================================
# Класс сканера
# ============================================================================

class PDNScanner:
    def __init__(self, max_workers: int = 8, detect_faces: bool = True):
        self.max_workers = max_workers
        self.detect_faces = detect_faces and YOLO_AVAILABLE
        self.results = []
        self.yolo_model = None
        
        if self.detect_faces:
            print("Инициализация YOLO для детекции лиц...")
            self.yolo_model = get_yolo_model()
            if self.yolo_model:
                print(f"✅ YOLO модель загружена на GPU: {GPU_AVAILABLE}")
            else:
                print("⚠️ Не удалось загрузить YOLO модель, детекция лиц отключена")
                self.detect_faces = False
    
    def scan_directory(self, directory_path: str, recursive: bool = True, sample_size: Optional[int] = None) -> pd.DataFrame:
        dir_path = Path(directory_path)
        if not dir_path.exists():
            raise FileNotFoundError(f"Директория не найдена: {directory_path}")
        
        print(f"Сбор файлов из: {dir_path.absolute()}")
        
        files = []
        if recursive:
            for ext in INCLUDE_EXTS:
                pattern = f"*.{ext}"
                files.extend(dir_path.rglob(pattern))
        else:
            for ext in INCLUDE_EXTS:
                pattern = f"*.{ext}"
                files.extend(dir_path.glob(pattern))
        
        files = list(set(files))
        file_paths = [str(f.absolute()) for f in files]
        
        print(f"Найдено файлов: {len(file_paths)}")
        
        if sample_size and sample_size < len(file_paths):
            file_paths = file_paths[:sample_size]
            print(f"Ограничено до: {len(file_paths)} файлов")
        
        if not file_paths:
            print("Нет файлов для анализа")
            return pd.DataFrame()
        
        return self._scan_with_threads(file_paths)
    
    def _scan_with_threads(self, file_paths: List[str]) -> pd.DataFrame:
        print(f"Анализ файлов (потоков: {self.max_workers})...")
        
        results = []
        lock = threading.Lock()
        processed = 0
        total = len(file_paths)
        
        def process_file(file_path: str):
            nonlocal processed
            try:
                file_size = os.path.getsize(file_path)
                if file_size > 50 * 1024 * 1024:  # 50 MB
                    with lock:
                        processed += 1
                    return {
                        'путь': file_path,
                        'категории_ПДн': 'нет',
                        'количество_находок': 0,
                        'УЗ': 'нет ПДн',
                        'формат_файла': Path(file_path).suffix.lower().lstrip('.'),
                        'success': False,
                        'error': 'Файл слишком большой (>50 MB)',
                        'size': file_size,
                        'time': get_file_creation_time_minus_2h(file_path),
                        'name': Path(file_path).name
                    }
                
                with open(file_path, 'rb') as f:
                    content = f.read()
                
                result = analyze_file_content(file_path, content, self.detect_faces, self.yolo_model)
                
                with lock:
                    processed += 1
                    if processed % 50 == 0:
                        print(f"  Прогресс: {processed}/{total} ({processed*100//total}%)")
                
                return result
            except Exception as e:
                with lock:
                    processed += 1
                    if processed % 50 == 0:
                        print(f"  Прогресс: {processed}/{total} ({processed*100//total}%)")
                return {
                    'путь': file_path,
                    'категории_ПДн': 'нет',
                    'количество_находок': 0,
                    'УЗ': 'нет ПДн',
                    'формат_файла': Path(file_path).suffix.lower().lstrip('.'),
                    'success': False,
                    'error': str(e)[:200],
                    'size': os.path.getsize(file_path) if os.path.exists(file_path) else 0,
                    'time': get_file_creation_time_minus_2h(file_path) if os.path.exists(file_path) else 'unknown',
                    'name': Path(file_path).name
                }
        
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = {executor.submit(process_file, fp): fp for fp in file_paths}
            for future in as_completed(futures):
                try:
                    result = future.result(timeout=30)
                    results.append(result)
                except Exception as e:
                    file_path = futures[future]
                    results.append({
                        'путь': file_path,
                        'категории_ПДн': 'нет',
                        'количество_находок': 0,
                        'УЗ': 'нет ПДн',
                        'формат_файла': Path(file_path).suffix.lower().lstrip('.'),
                        'success': False,
                        'error': f'Таймаут или ошибка: {str(e)[:100]}',
                        'size': os.path.getsize(file_path) if os.path.exists(file_path) else 0,
                        'time': get_file_creation_time_minus_2h(file_path) if os.path.exists(file_path) else 'unknown',
                        'name': Path(file_path).name
                    })
        
        print(f"Анализ завершен. Обработано {len(results)} файлов")
        
        df = pd.DataFrame(results)
        
        if not df.empty:
            for col in ['путь', 'категории_ПДн', 'УЗ', 'формат_файла', 'error', 'time', 'name']:
                if col in df.columns:
                    df[col] = df[col].fillna('').astype(str)
                    df[col] = df[col].replace(['nan', 'None', 'NaN'], '')
            
            for col in ['количество_находок', 'size']:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
            
            if 'success' in df.columns:
                df['success'] = df['success'].fillna(False).astype(bool)
        
        # Выводим результаты в формате size,time,name (БЕЗ подчеркиваний)
        if not df.empty:
            print("\nРезультаты в формате size,time,name:")
            print("size,time,name")
            for _, row in df.iterrows():
                size_val = int(row.get('size', 0))
                time_val = str(row.get('time', ''))  # БЕЗ замены пробелов на подчеркивания
                name_val = str(row.get('name', ''))
                print(f"{size_val},{time_val},{name_val}")
            
            files_with_pdn = df[df['количество_находок'] > 0]
            if not files_with_pdn.empty:
                print(f"\nНайдено {len(files_with_pdn)} файлов с ПДн")
        
        return df
    
    def save_report_csv(self, df: pd.DataFrame, output_path: str) -> str:
        """Сохранение CSV только с колонками size,time,name"""
        columns = ['size', 'time', 'name']
        
        if df.empty:
            empty_df = pd.DataFrame(columns=columns)
            empty_df.to_csv(output_path, index=False, encoding='utf-8-sig')
            print(f"CSV отчет сохранен: {output_path}")
            return output_path
        
        available_cols = [col for col in columns if col in df.columns]
        report_df = df[available_cols].copy()
        
        # Очистка от NaN
        for col in ['time', 'name']:
            if col in report_df.columns:
                report_df[col] = report_df[col].fillna('').astype(str)
                report_df[col] = report_df[col].replace(['nan', 'None', 'NaN'], '')
        
        if 'size' in report_df.columns:
            report_df['size'] = pd.to_numeric(report_df['size'], errors='coerce').fillna(0).astype(int)
        
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        report_df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"CSV отчет сохранен: {output_path}")
        return output_path
    
    def get_statistics(self, df: pd.DataFrame) -> Dict[str, Any]:
        if df.empty:
            return {
                'total_files': 0,
                'successful': 0,
                'failed': 0,
                'files_with_pdn': 0,
                'protection_levels': {},
                'total_pdn_hits': 0
            }
        
        total = len(df)
        success = df['success'].sum() if 'success' in df.columns else len(df)
        with_pdn = (df['количество_находок'] > 0).sum() if 'количество_находок' in df.columns else 0
        
        uz_stats = {}
        if 'УЗ' in df.columns:
            uz_series = df['УЗ'].fillna('нет ПДн').astype(str)
            uz_stats = uz_series.value_counts().to_dict()
        
        total_hits = int(df['количество_находок'].fillna(0).sum()) if 'количество_находок' in df.columns else 0
        
        return {
            'total_files': total,
            'successful': int(success),
            'failed': total - int(success),
            'files_with_pdn': int(with_pdn),
            'protection_levels': uz_stats,
            'total_pdn_hits': total_hits
        }

# ============================================================================
# Точка входа
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print("СКАНЕР ПЕРСОНАЛЬНЫХ ДАННЫХ")
    print("="*60)
    print(f"GPU доступен: {GPU_AVAILABLE}")
    if GPU_AVAILABLE:
        print(f"GPU: {GPU_NAME}")
    print

СКАНЕР ПЕРСОНАЛЬНЫХ ДАННЫХ
GPU доступен: True
GPU: CUDA (NVIDIA GeForce RTX 4060 Laptop GPU)


In [ ]:
# ============================================================
# ЗАПУСК СКАНЕРА И СОХРАНЕНИЕ В CSV (size,time,name)
# Сохраняются ТОЛЬКО файлы с персональными данными
# ============================================================

import csv

SCAN_DIRECTORY = "."          # <-- укажите папку для сканирования
OUTPUT_CSV     = "result.csv" # <-- путь к выходному файлу

scanner = PDNScanner(max_workers=8, detect_faces=False)
df = scanner.scan_directory(SCAN_DIRECTORY, recursive=True, sample_size=None)

if not df.empty:
    # --- Оставляем только файлы С персональными данными ---
    pdn_df = df[df["количество_находок"] > 0].copy()

    print(f"Всего файлов просканировано : {len(df)}")
    print(f"Файлов с ПДн               : {len(pdn_df)}")

    if pdn_df.empty:
        print("Персональные данные не обнаружены.")
    else:
        # --- Формируем итоговую таблицу: только size, time, name ---
        out = pdn_df[["size", "time", "name"]].copy()

        # Жёсткая очистка NaN на всех уровнях
        NAN_VALS = ["nan", "NaN", "None", "none", "", "NaT"]
        out["size"] = pd.to_numeric(out["size"], errors="coerce").fillna(0).astype(int)
        out["time"] = out["time"].fillna("unknown").astype(str).replace(NAN_VALS, "unknown")
        out["name"] = out["name"].fillna("unknown").astype(str).replace(NAN_VALS, "unknown")

        # Финальная проверка — убедиться что ни одного NaN не осталось
        assert not out.isnull().any().any(), "В данных остались NaN!"
        assert (out["name"] != "unknown").all(), "Пустые имена файлов в результате!"

        # --- Сохраняем result.csv ---
        out = out.sort_values("size", ascending=False).reset_index(drop=True)
        out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig",
                   quoting=csv.QUOTE_MINIMAL)

        print(f"CSV сохранён: {OUTPUT_CSV}  ({len(out)} строк)")
        print("Первые строки:")
        print("size,time,name")
        for _, row in out.head(10).iterrows():
            print(f"{row['size']},{row['time']},{row['name']}")
else:
    print("Файлы не найдены или директория пуста.")


Сбор файлов из: c:\Users\rulan\Downloads\SamsungHacaton
Найдено файлов: 3307
Ограничено до: 100 файлов
Анализ файлов (потоков: 8)...
  Прогресс: 50/100 (50%)
  Прогресс: 100/100 (100%)
Анализ завершен. Обработано 100 файлов

Результаты в формате size,time,name:
size,time,name
511872,sep 26 15:54,%D0%9F15_%D0%9C%D0%A3%D0%90%D0%9C_%D0%A6%D0%9A.pdf
544475,sep 26 11:25,4.1.3.pdf
141497,apr 29 03:54,page_64.html
394254,sep 26 15:15,ekol_prirod.pdf
27601,apr 29 03:53,1929784_300.jpg
618522,apr 29 03:53,page_336.html
324405,apr 29 03:53,1065550_1000.png
56278,sep 26 21:11,0001142383.tif
434142,sep 26 21:11,2501298013.tif
993,sep 26 09:44,27_03_01.pdf
300,sep 26 12:06,politika-cookies.pdf
615015,sep 26 12:31,prikaz_34n_ot_29.08.2018.pdf
6261936,sep 26 16:16,09.04.01 Информатика и вычислительная техника.pdf
83971,sep 26 12:32,Prikaz_П196.pdf
10521,sep 26 15:00,00860-PRONTUARIO_Difusion_Digital.pdf
1238287,sep 26 15:51,110302_B_3_17102017.pdf
31328,sep 26 14:18,P1676_16.07.2024.pdf
402109,apr 29